# Notebook 03: SQLインジェクション

SQLインジェクションは「世界で最も有名なWebセキュリティ脆弱性」の一つです。
OWASP Top 10 に20年以上ランクインし続けており、今でも実際の攻撃に使われています。

## 学習目標
1. SQL の基本を理解する（知らなくてもOK！）
2. SQLインジェクションがなぜ起きるかを理解する
3. 実際に labs/ のアプリで攻撃を試す
4. 防御方法（パラメータ化クエリ）を実装する

---

⚠️ **この知識は labs/vulnerable_app の練習環境のみで使ってください**

## 1. SQL の基本（5分で理解する）

データベースはデータを「表」で管理しています。SQL はその表を操作する言語です。

### users テーブルの例
```
| id | username | password    | role  |
|----|----------|-------------|-------|
| 1  | alice    | password123 | user  |
| 2  | bob      | qwerty      | user  |
| 3  | admin    | admin_secret| admin |
```

### よく使うSQL文
```sql
-- 全データを取得
SELECT * FROM users;

-- 条件で絞り込む
SELECT * FROM users WHERE username = 'alice';

-- ログイン確認（よくある実装）
SELECT * FROM users WHERE username = 'alice' AND password = 'password123';
```

ログインのSQL文が **1行以上返ってきたら** ログイン成功という設計が多いです。

In [ ]:
# SQLite で実際のデータベース操作を体験
import sqlite3

# メモリ上にデータベースを作成（ファイルに保存しない）
conn = sqlite3.connect(":memory:")
c = conn.cursor()

# テーブル作成とデータ挿入
c.execute("""
    CREATE TABLE users (
        id INTEGER PRIMARY KEY,
        username TEXT,
        password TEXT,
        role TEXT
    )
""")
c.execute("INSERT INTO users VALUES (1, 'alice', 'password123', 'user')")
c.execute("INSERT INTO users VALUES (2, 'bob', 'qwerty', 'user')")
c.execute("INSERT INTO users VALUES (3, 'admin', 'admin_secret', 'admin')")
conn.commit()

# 正常なログイン
print("=== 正常なログイン ===")
result = c.execute(
    "SELECT * FROM users WHERE username='alice' AND password='password123'"
).fetchall()
print(f"結果: {result}")
print(f"ログイン {'成功' if result else '失敗'}")
print()

# パスワード間違い
print("=== パスワード間違い ===")
result = c.execute(
    "SELECT * FROM users WHERE username='alice' AND password='wrong'"
).fetchall()
print(f"結果: {result}")
print(f"ログイン {'成功' if result else '失敗'}")

## 2. SQLインジェクションとは

多くのアプリは、ユーザーが入力したデータをSQL文の中に直接埋め込みます。

### 脆弱なコードの例
```python
# ❌ 危険なコード
username = request.form['username']  # ユーザーの入力
password = request.form['password']

sql = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
result = db.execute(sql)
```

### 正常な入力の場合
```
username = alice
password = password123

生成されるSQL:
SELECT * FROM users WHERE username='alice' AND password='password123'
```

### 攻撃的な入力の場合 😈
```
username = admin' --
password = （なんでもよい）

生成されるSQL:
SELECT * FROM users WHERE username='admin' --' AND password='なんでも'
                                          ↑
                                   SQLのコメント記号！
                                   これ以降は無視される
```

パスワードチェックが無効化され、admin としてログインできてしまいます！

In [ ]:
# SQLインジェクション攻撃のシミュレーション
import sqlite3

conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE users (id INTEGER, username TEXT, password TEXT, role TEXT)")
c.execute("INSERT INTO users VALUES (1, 'alice', 'password123', 'user')")
c.execute("INSERT INTO users VALUES (2, 'admin', 'super_secret_pw', 'admin')")
conn.commit()

def vulnerable_login(username: str, password: str):
    """脆弱なログイン関数（SQLiに注意！）"""
    # 🚨 ユーザー入力をそのまま埋め込んでいる
    sql = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    print(f"実行SQL: {sql}")
    try:
        result = c.execute(sql).fetchall()
        if result:
            return f"✅ ログイン成功: {result[0]}"
        return "❌ ログイン失敗"
    except Exception as e:
        return f"⚠️ SQLエラー: {e}"

print("=== 正常なログイン ===")
print(vulnerable_login("alice", "password123"))
print()

print("=== 攻撃1: コメントによるbypass ===")
print(vulnerable_login("admin' --", "なんでもよい"))
print()

print("=== 攻撃2: OR条件の注入 ===")
print(vulnerable_login("' OR '1'='1' --", ""))
print()

print("=== 攻撃3: 別テーブルへのアクセス（UNION攻撃） ===")
# まず秘密テーブルを作る
c.execute("CREATE TABLE secret_data (id INTEGER, data TEXT)")
c.execute("INSERT INTO secret_data VALUES (1, 'TOP_SECRET_FLAG_12345')")
conn.commit()

# UNION を使って別テーブルのデータを取得
print(vulnerable_login("' UNION SELECT id, data, '', '' FROM secret_data --", ""))

## 3. 実際に labs/ アプリで試してみよう

以下の手順でラボアプリを起動して、実際のWebブラウザで攻撃を体験します。

```bash
# ターミナルで実行
cd labs/vulnerable_app
pip install flask
python app.py
```

ブラウザで `http://localhost:5000` を開く

### Lab 1: 商品検索のSQLi

通常の検索: `リンゴ`
→ リンゴが表示される

攻撃入力1: `%' OR '1'='1`
→ 全商品が表示される（秘密の商品も！）

攻撃入力2: `%' UNION SELECT id, flag, hint, '' FROM flags --`
→ フラグテーブルの中身が見える！

### Lab 2: ログインbypass

正常なログイン: username=`alice`, password=`password123`

攻撃: username=`admin' --`, password=`(何でもよい)`
→ admin のパスワードを知らずにログインできる！

In [ ]:
# Python から labs アプリに対してSQLiを試す（アプリ起動後に実行）
# ※ labs/vulnerable_app/app.py を起動してから実行してください

import requests

BASE_URL = "http://localhost:5000"

def test_search(query: str):
    """商品検索エンドポイントをテスト"""
    try:
        r = requests.get(f"{BASE_URL}/lab/sqli/search", params={"q": query}, timeout=3)
        # 結果のHTML長さで件数を大まかに確認
        return f"HTTP {r.status_code}, レスポンスサイズ: {len(r.text)} bytes"
    except requests.ConnectionError:
        return "⚠️ アプリが起動していません。先に python app.py を実行してください"

print("通常検索 'リンゴ':", test_search("リンゴ"))
print("攻撃 '% OR 1=1--':", test_search("%' OR '1'='1' --"))
print()
print("ブラウザで http://localhost:5000/lab/sqli/search にアクセスして")
print("実際の表示を確認してみましょう！")

## 4. 防御方法：パラメータ化クエリ

SQLインジェクションを防ぐ方法は明確で、**パラメータ化クエリ（プリペアドステートメント）** を使うことです。

文字列の結合でSQLを作るのをやめ、値をSQL文とは別に渡します。

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE users (id INTEGER, username TEXT, password TEXT, role TEXT)")
c.execute("INSERT INTO users VALUES (1, 'alice', 'password123', 'user')")
c.execute("INSERT INTO users VALUES (2, 'admin', 'super_secret', 'admin')")
conn.commit()

def safe_login(username: str, password: str):
    """
    ✅ 安全なログイン関数
    パラメータ化クエリ（?プレースホルダー）を使用
    """
    # SQL文とパラメータを分けて渡す
    sql = "SELECT * FROM users WHERE username=? AND password=?"
    result = c.execute(sql, (username, password)).fetchall()
    # ↑ ここが重要！文字列結合ではなく、タプルで渡している
    if result:
        return f"✅ ログイン成功: {result[0]}"
    return "❌ ログイン失敗"

print("=== 安全なバージョンでの攻撃テスト ===")
print()

print("正常なログイン:")
print(safe_login("alice", "password123"))
print()

print("攻撃1 (admin' --):") 
print(safe_login("admin' --", "なんでも"))
print("→ SQLiが効かない！'admin' -- という文字列としてそのまま検索される")
print()

print("攻撃2 (OR 1=1):") 
print(safe_login("' OR '1'='1' --", ""))
print("→ 攻撃文字列はそのまま文字列として扱われ、一致するユーザーがいないので失敗")

In [ ]:
# なぜパラメータ化クエリは安全なのか？
# 仕組みを可視化する

print("=== 脆弱なコードの処理 ===")
username = "admin' --"
password = "なんでも"

sql_vulnerable = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
print(f"生成されるSQL:\n{sql_vulnerable}")
print("→ SQL構造が書き換えられている！")
print()

print("=== 安全なコードの処理 ===")
sql_safe_template = "SELECT * FROM users WHERE username=? AND password=?"
params = (username, password)
print(f"SQLテンプレート: {sql_safe_template}")
print(f"パラメータ: {params}")
print()
print("実際にDBに渡されるのは:")
print("  SQL: SELECT * FROM users WHERE username=? AND password=?")
print(f"  値1: {repr(username)}  ← SQL構造を変えられない")
print(f"  値2: {repr(password)}")
print()
print("DB側でエスケープ処理が行われるため、")
print("ユーザー入力がSQL命令として解釈されることはない")

## 5. 他のフレームワークでの防御方法

### Python SQLAlchemy（ORM）
```python
# ✅ 安全（ORMが自動でエスケープ）
user = db.session.query(User).filter_by(username=username, password=password).first()
```

### Node.js + MySQL2
```javascript
// ✅ 安全
const [rows] = await connection.execute(
  'SELECT * FROM users WHERE username = ? AND password = ?',
  [username, password]
);
```

### PHP PDO
```php
// ✅ 安全
$stmt = $pdo->prepare('SELECT * FROM users WHERE username = ? AND password = ?');
$stmt->execute([$username, $password]);
```

どの言語でも **文字列結合を避けて `?` プレースホルダー** を使うのが基本です。

## まとめ

| | 脆弱なコード | 安全なコード |
|-|------------|------------|
| SQL作成方法 | 文字列結合 `f"...{username}..."` | パラメータ化 `"...?..."` + タプル |
| 攻撃 | `' OR '1'='1` が効く | 効かない（文字列として扱われる） |
| 一言 | ユーザーを信頼している | ユーザーを信頼しない |

**「ユーザーの入力を信頼しない」** — これがWebセキュリティの最大原則です。

## 演習

1. labs/vulnerable_app を起動して、Lab 1 の UNION 攻撃でフラグを取得してみよう
2. `safe_login()` のコードを修正して、ロールが `admin` のユーザーのみ成功するようにしよう
3. 以下の URL は SQL インジェクションに脆弱か？理由を説明してみよう：
   - `/search?id=1`
   - `/user/alice`
   - `/products?category=fruit&sort=price`